In [ ]:
import json
from sklearn.model_selection import train_test_split

In [ ]:
# with open('Data/data_augmented.json', 'r') as f:
#     data = json.load(f)
# train_data, test_data = train_test_split(data, test_size=0.1, random_state=42)
# train_data, val_data = train_test_split(train_data, test_size=0.1, random_state=42)

# with open('Data/train_data.json', 'w') as f:
#     json.dump(train_data, f, indent=4)
# with open('Data/val_data.json', 'w') as f:
#     json.dump(val_data, f, indent=4)
# with open('Data/test_data.json', 'w') as f:
#     json.dump(test_data, f, indent=4)

In [ ]:
# len(data)

In [ ]:
with open('Data/train_data.json', 'r') as f:
    train_data = json.load(f)
with open('Data/val_data.json', 'r') as f:
    val_data = json.load(f)
with open('Data/test_data.json', 'r') as f:
    test_data = json.load(f)

In [ ]:
test_data.sort(key=lambda x: len(x['case_details']), reverse=True)
test_data[2]

In [ ]:
def prep_data_for_ft(data, filename, include_statute_details=False): 
    with open('Data/' + filename + '.jsonl', 'w', encoding='utf-8') as f:
        for item in data:
            bail_type = "Applicant applied for " + item['bail_type'] + ". "
            age = "The age of the applicant/s is/are " + str(item['ages']) + ". " if item['age_available'] else "Age not available. "
            health = "The health condition of the applicant is " + item['health_condition'] + " " if item['health_condition'] is not None else "none. "
            past = "Past criminal records for the applicant "+ ("exist." if item['past_criminal_record_exists'] else "do not exist.") + " "
            statutes = "The relevant statutes are: " + ", ".join(item['statutes']) + ". "
            custody = "The applicant is in custody for " + str(item['days_in_custody']) + " days. " if item['days_in_custody'] is not None else ""
            case_details = item['case_details'] + " " if item['case_details'] is not None else ""
            outcome = "The outcome of the case is " + item['outcome'] + "."
            reasoning = item['reasoning'] if item['reasoning'] is not None else ""
            statutes_info = " ".join(item['statute_details']) + " " if item['statute_details'] is not None else ""

            user_input = bail_type + age + health + past + statutes + custody + case_details + outcome + (" " + statutes_info if include_statute_details else "")
            output = reasoning

            record = {
                "messages": [
                    {"role": "user", "content": user_input.strip()},
                    {"role": "assistant", "content": output.strip()}
                ]
            }
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

In [ ]:
prep_data_for_ft(train_data, 'train_ft_with_statutes', include_statute_details=True)
prep_data_for_ft(val_data, 'val_ft_with_statutes', include_statute_details=True)
prep_data_for_ft(test_data, 'test_ft_with_statutes', include_statute_details=True)

In [ ]:
import json

with open("Data/train_ft_with_statutes.jsonl") as f:
    for i, line in enumerate(f, 1):
        try:
            obj = json.loads(line)
            assert "messages" in obj
            assert len(obj["messages"]) >= 2
        except Exception as e:
            print(f"Error on line {i}: {e}")